In [4]:
! pip install fastapi uvicorn tensorflow emoji nltk


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 14.4 MB/s eta 0:00:00


In [6]:
# pip install fastapi uvicorn tensorflow emoji nltk
# Run: uvicorn server:app --reload
# Docs: http://localhost:8000/docs

import re, string, pickle, json
import numpy as np
import emoji
import nltk
from nltk.corpus import stopwords
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from fastapi import FastAPI
from pydantic import BaseModel

nltk.download("stopwords", quiet=True)

# ── Load artifacts ─────────────────────────────────────────────────────────────
SAVE_DIR = "/content/drive/MyDrive/CollegeProjects/nlp_project/saved_model"   # copy this folder next to server.py

model     = load_model(f"{SAVE_DIR}/sentiment_model.keras")
tokenizer = pickle.load(open(f"{SAVE_DIR}/tokenizer.pkl", "rb"))
le        = pickle.load(open(f"{SAVE_DIR}/label_encoder.pkl", "rb"))
MAX_LEN   = json.load(open(f"{SAVE_DIR}/config.json"))["max_len"]

STOP_WORDS = set(stopwords.words("english")) - {"not", "no", "never", "nor"}

# ── Text cleaning (same as training) ──────────────────────────────────────────
def clean(text):
    text = emoji.demojize(str(text).lower(), delimiters=(" ", " "))
    text = re.sub(r"#(\w+)", r"\1", text)           # #happy → happy
    text = re.sub(r"@\w+|http\S+", "", text)         # remove mentions & URLs
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)       # soooo → soo
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = " ".join(w for w in text.split() if w not in STOP_WORDS)
    return text.strip()

# ── FastAPI ────────────────────────────────────────────────────────────────────
app = FastAPI(title="Tweet Sentiment API")

class TweetIn(BaseModel):
    text: str

@app.post("/predict")
def predict(req: TweetIn):
    cleaned = clean(req.text)
    seq     = tokenizer.texts_to_sequences([cleaned])
    padded  = pad_sequences(seq, maxlen=MAX_LEN, padding="post", truncating="post")
    proba   = model.predict(padded, verbose=0)[0]
    label   = le.inverse_transform([np.argmax(proba)])[0]
    return {
        "sentiment":  label,
        "confidence": {c: round(float(p), 4) for c, p in zip(le.classes_, proba)}
    }

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive
